<a href="https://colab.research.google.com/github/settuce-c/Watermarking/blob/main/Watermark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Zahra Zahidah Putri Ruchwijaya

NIM: 18224058


Tugas Sistem Multimedia

Watermarking

In [ ]:
# Instalasi Library

!pip install numpy matplotlib Pillow -q
print("Library siap!")

In [ ]:
# Import Library

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import io
import os

# Agar hasil random bisa direproduksi
np.random.seed(42)

print("Import berhasil!")
print(f"NumPy  : {np.__version__}")
print(f"Pillow : {Image.__version__}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import io
import os

# Agar hasil random bisa direproduksi
np.random.seed(42)

print("✅ Import berhasil!")
print(f"NumPy  : {np.__version__}")
print(f"Pillow : {Image.__version__}")

# ============================================================
# FUNGSI DCT & IDCT MANUAL (tanpa library bawaan seperti scipy.fft)
# ============================================================

def dct2d_manual(block):
    """
    DCT-II 2D untuk blok 8x8 piksel.
    Menggunakan matriks transformasi DCT untuk efisiensi.

    Rumus: F = D @ block @ D.T
    Di mana D adalah matriks DCT 8x8.
    """
    N = 8
    # Buat matriks DCT (N x N)
    D = np.zeros((N, N))
    for u in range(N):
        for x in range(N):
            if u == 0:
                D[u, x] = np.sqrt(1.0 / N)
            else:
                D[u, x] = np.sqrt(2.0 / N) * np.cos((2 * x + 1) * u * np.pi / (2 * N))

    # Transformasi 2D: F = D @ block @ D.T
    return D @ block @ D.T


def idct2d_manual(block):
    """
    Inverse DCT-II 2D untuk blok 8x8.
    Menggunakan matriks yang sama (D^T @ F @ D karena D orthogonal).
    """
    N = 8
    D = np.zeros((N, N))
    for u in range(N):
        for x in range(N):
            if u == 0:
                D[u, x] = np.sqrt(1.0 / N)
            else:
                D[u, x] = np.sqrt(2.0 / N) * np.cos((2 * x + 1) * u * np.pi / (2 * N))

    # IDCT: block_asli = D.T @ F @ D
    return D.T @ block @ D


def apply_dct_to_image(img_array):
    """
    Terapkan DCT manual ke seluruh citra per blok 8x8.
    Citra akan di-pad agar dimensinya kelipatan 8.
    """
    h, w = img_array.shape
    # Pad agar kelipatan 8
    pad_h = (8 - h % 8) % 8
    pad_w = (8 - w % 8) % 8
    padded = np.pad(img_array, ((0, pad_h), (0, pad_w)), mode='edge').astype(float)

    H, W = padded.shape
    dct_img = np.zeros_like(padded)

    for i in range(0, H, 8):
        for j in range(0, W, 8):
            block = padded[i:i+8, j:j+8] - 128.0  # Level shifting
            dct_img[i:i+8, j:j+8] = dct2d_manual(block)

    return dct_img, (h, w)  # Kembalikan juga dimensi asli


def apply_idct_to_image(dct_img, original_shape):
    """
    Terapkan IDCT manual ke seluruh citra (domain frekuensi → spasial).
    """
    H, W = dct_img.shape
    img_out = np.zeros_like(dct_img)

    for i in range(0, H, 8):
        for j in range(0, W, 8):
            block = dct_img[i:i+8, j:j+8]
            img_out[i:i+8, j:j+8] = idct2d_manual(block) + 128.0  # Inverse level shifting

    # Crop ke dimensi asli dan clip ke [0, 255]
    h, w = original_shape
    img_out = np.clip(img_out[:h, :w], 0, 255).astype(np.uint8)
    return img_out


# === VERIFIKASI DCT ===
test_block = np.random.randint(0, 256, (8, 8), dtype=np.uint8)
dct_result = dct2d_manual(test_block - 128)
reconstructed = idct2d_manual(dct_result) + 128
error = np.max(np.abs(test_block - reconstructed))

print("🔬 Verifikasi DCT Manual:")
print(f"   Max rekonstruksi error: {error:.6f}")
print(f"   Status: {'✅ LULUS (error < 1e-9)' if error < 1e-9 else '❌ Ada masalah!'}")

In [ ]:
# ============================================================
# TABEL KUANTISASI JPEG STANDAR (JPEG Standard Annex K)
# ============================================================

# Tabel kuantisasi luminance (kanal Y/grayscale)
JPEG_QUANT_TABLE_LUMA = np.array([
    [16, 11, 10, 16, 24,  40,  51,  61],
    [12, 12, 14, 19, 26,  58,  60,  55],
    [14, 13, 16, 24, 40,  57,  69,  56],
    [14, 17, 22, 29, 51,  87,  80,  62],
    [18, 22, 37, 56, 68, 109, 103,  77],
    [24, 35, 55, 64, 81, 104, 113,  92],
    [49, 64, 78, 87,103, 121, 120, 101],
    [72, 92, 95, 98,112, 100, 103,  99]
], dtype=float)


def get_quant_table(quality_factor):
    """
    Hitung tabel kuantisasi berdasarkan Quality Factor (QF).
    Rumus standar JPEG:
      - QF < 50: scale = 5000 / QF
      - QF >= 50: scale = 200 - 2*QF
    Nilai hasil di-clip ke rentang [1, 255].
    """
    if quality_factor <= 0:
        quality_factor = 1
    if quality_factor >= 100:
        return np.ones((8, 8))

    if quality_factor < 50:
        scale = 5000.0 / quality_factor
    else:
        scale = 200.0 - 2.0 * quality_factor

    quant_table = np.floor((JPEG_QUANT_TABLE_LUMA * scale + 50) / 100)
    quant_table = np.clip(quant_table, 1, 255)
    return quant_table


def quantize_block(dct_block, quant_table):
    """Kuantisasi koefisien DCT: pembulatan hasil bagi dengan tabel kuantisasi."""
    return np.round(dct_block / quant_table)


def dequantize_block(quant_block, quant_table):
    """De-kuantisasi: kalikan kembali dengan tabel kuantisasi (pemulihan perkiraan)."""
    return quant_block * quant_table


def jpeg_compress_decompress(img_array, quality_factor):
    """
    Simulasi kompresi & dekompresi JPEG manual.
    Langkah:
      1. DCT pada setiap blok 8x8
      2. Kuantisasi dengan tabel QF
      3. De-kuantisasi (simulasi decode)
      4. IDCT → kembalikan ke domain spasial
    """
    quant_table = get_quant_table(quality_factor)

    h, w = img_array.shape
    pad_h = (8 - h % 8) % 8
    pad_w = (8 - w % 8) % 8
    padded = np.pad(img_array, ((0, pad_h), (0, pad_w)), mode='edge').astype(float)
    H, W = padded.shape

    result = np.zeros_like(padded)

    for i in range(0, H, 8):
        for j in range(0, W, 8):
            block = padded[i:i+8, j:j+8] - 128.0
            # DCT
            dct_block = dct2d_manual(block)
            # Kuantisasi
            q_block = quantize_block(dct_block, quant_table)
            # De-kuantisasi
            dq_block = dequantize_block(q_block, quant_table)
            # IDCT
            result[i:i+8, j:j+8] = idct2d_manual(dq_block) + 128.0

    return np.clip(result[:h, :w], 0, 255).astype(np.uint8)


# === VISUALISASI TABEL KUANTISASI ===
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
qfs = [95, 75, 50, 10]
for ax, qf in zip(axes, qfs):
    qt = get_quant_table(qf)
    im = ax.imshow(qt, cmap='hot', aspect='auto')
    ax.set_title(f'QF = {qf}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Frekuensi Horizontal')
    ax.set_ylabel('Frekuensi Vertikal')
    plt.colorbar(im, ax=ax)
    # Anotasi nilai
    for r in range(8):
        for c in range(8):
            ax.text(c, r, f'{int(qt[r,c])}', ha='center', va='center',
                    fontsize=6, color='cyan' if qt[r,c] > 50 else 'black')

plt.suptitle('Tabel Kuantisasi JPEG untuk Berbagai Quality Factor\n(nilai besar = lebih banyak detail hilang)',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('quant_tables.png', dpi=100, bbox_inches='tight')
plt.show()
print("💡 Perhatikan: QF rendah = nilai kuantisasi besar = banyak koefisien DCT yang menjadi 0!")

In [ ]:
# ============================================================
# KONFIGURASI FOTO
# Ubah USE_CUSTOM_PHOTO = True dan isi PHOTO_PATH untuk pakai foto sendiri
# ============================================================
USE_CUSTOM_PHOTO = True
PHOTO_PATH = '/content/Photo_Zahra Zahidah Putri R.jpg'   # Ganti dengan path foto kamu

IMG_SIZE = 256  # Ukuran citra yang digunakan (kelipatan 8)


def create_synthetic_photo(size=256):
    """
    Buat foto sintetis menyerupai foto portrait dengan gradasi warna dan tekstur.
    Digunakan jika tidak ada foto yang diupload.
    """
    img = np.zeros((size, size), dtype=np.uint8)

    # Latar belakang gradasi
    for i in range(size):
        img[i, :] = int(30 + 40 * i / size)

    # Lingkaran wajah
    cx, cy, r = size//2, size//2 - size//10, size//4
    Y, X = np.ogrid[:size, :size]
    mask_face = (X - cx)**2 + (Y - cy)**2 <= r**2
    img[mask_face] = 200

    # Rambut
    mask_hair = (X - cx)**2 + (Y - (cy - r//2))**2 <= (r + 10)**2
    mask_hair &= (Y < cy - r//3)
    img[mask_hair] = 60

    # Mata kiri & kanan
    for ex in [cx - r//3, cx + r//3]:
        ey = cy - r//6
        mask_eye = (X - ex)**2 + (Y - ey)**2 <= (r//8)**2
        img[mask_eye] = 40

    # Hidung
    mask_nose = (X - cx)**2 + (Y - (cy + r//8))**2 <= (r//12)**2
    img[mask_nose] = 160

    # Mulut
    mask_mouth = (X - cx)**2 + (Y - (cy + r//3))**2 <= (r//5)**2
    mask_mouth &= Y > cy + r//3
    img[mask_mouth] = 100

    # Badan/baju
    mask_body = (Y > cy + r) & (Y < size - 1)
    mask_body &= (X > cx - r//1.2) & (X < cx + r//1.2)
    img[mask_body.astype(bool)] = 80

    # Tambahkan sedikit noise agar tekstur lebih realistis
    noise = np.random.normal(0, 8, img.shape).astype(int)
    img = np.clip(img.astype(int) + noise, 0, 255).astype(np.uint8)

    return img


# Load atau buat foto
if USE_CUSTOM_PHOTO and os.path.exists(PHOTO_PATH):
    pil_img = Image.open(PHOTO_PATH).convert('L')  # Konversi ke grayscale
    pil_img = pil_img.resize((IMG_SIZE, IMG_SIZE))
    original_img = np.array(pil_img)
    print(f"✅ Foto '{PHOTO_PATH}' berhasil dimuat!")
else:
    original_img = create_synthetic_photo(IMG_SIZE)
    print("📸 Menggunakan foto sintetis (portrait sederhana).")
    print("💡 Untuk pakai foto sendiri: set USE_CUSTOM_PHOTO=True dan isi PHOTO_PATH")

# Tampilkan foto
plt.figure(figsize=(5, 5))
plt.imshow(original_img, cmap='gray', vmin=0, vmax=255)
plt.title(f'Foto Asli (Grayscale, {IMG_SIZE}×{IMG_SIZE} piksel)', fontsize=13)
plt.colorbar(label='Intensitas Piksel (0=hitam, 255=putih)')
plt.axis('off')
plt.tight_layout()
plt.savefig('foto_asli.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"📐 Ukuran foto: {original_img.shape}, dtype: {original_img.dtype}")
print(f"📊 Nilai piksel: min={original_img.min()}, max={original_img.max()}, mean={original_img.mean():.1f}")

In [ ]:
# ============================================================
# PEMBUATAN WATERMARK
# ============================================================

# Hitung jumlah blok 8x8 dalam citra
N_BLOCKS_H = IMG_SIZE // 8  # Jumlah blok arah vertikal
N_BLOCKS_W = IMG_SIZE // 8  # Jumlah blok arah horizontal
WM_SIZE = N_BLOCKS_H * N_BLOCKS_W  # Total blok = panjang watermark

print(f"📦 Jumlah blok 8×8: {N_BLOCKS_H} × {N_BLOCKS_W} = {WM_SIZE} blok")
print(f"→ Watermark berukuran: {WM_SIZE} bit/nilai")


# --- WATERMARK 1: BINARY ---
def create_binary_watermark(size):
    """
    Buat binary watermark berupa pola biner bermakna.
    Nilai: 0 atau 1, lalu dipetakan ke -1 dan +1 untuk disisipkan di DCT.
    """
    wm = np.zeros(size, dtype=int)
    # Buat pola checkboard di bagian tengah
    side = int(np.sqrt(size))
    wm_2d = np.zeros((side, side), dtype=int)
    # Inisial 'WM' dalam grid biner 32x32
    # 'W'
    for r in range(4, 20):
        wm_2d[r, 2] = 1
        wm_2d[r, 7] = 1
        wm_2d[r, 12] = 1
    for c in range(2, 13):
        wm_2d[19, c] = 1
    wm_2d[14, 4] = wm_2d[14, 5] = wm_2d[14, 9] = wm_2d[14, 10] = 1
    # 'M'
    for r in range(4, 20):
        wm_2d[r, 16] = 1
        wm_2d[r, 28] = 1
    for c in range(16, 29):
        wm_2d[4, c] = 1
    for r in range(4, 14):
        wm_2d[r, 22] = 1
    return wm_2d.flatten() * 2 - 1  # Konversi 0/1 → -1/+1


# --- WATERMARK 2: RANDOM ---
def create_random_watermark(size, seed=2024):
    """
    Buat random watermark dengan seed tertentu (kunci rahasia).
    Nilai: ±1 (distribusi seragam).
    Seed ini HARUS disimpan untuk bisa mengekstrak watermark nanti.
    """
    rng = np.random.default_rng(seed)
    wm = rng.choice([-1, 1], size=size)
    return wm


# Buat kedua watermark
SEED_RANDOM_WM = 2024  # Simpan ini sebagai 'kunci rahasia'!
binary_watermark = create_binary_watermark(WM_SIZE)
random_watermark = create_random_watermark(WM_SIZE, seed=SEED_RANDOM_WM)

print(f"\n🔐 Binary Watermark : {WM_SIZE} nilai, unik: {np.unique(binary_watermark)}")
print(f"🎲 Random Watermark : {WM_SIZE} nilai, unik: {np.unique(random_watermark)}")
print(f"   Distribusi random: {np.sum(random_watermark==1)} nilai +1, {np.sum(random_watermark==-1)} nilai -1")

# Tampilkan kedua watermark dalam bentuk 2D
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Binary watermark
wm_bin_2d = binary_watermark.reshape(N_BLOCKS_H, N_BLOCKS_W)
axes[0].imshow(wm_bin_2d, cmap='RdBu', vmin=-1, vmax=1, aspect='auto')
axes[0].set_title('Binary Watermark\n(pola inisial "WM")', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Kolom Blok')
axes[0].set_ylabel('Baris Blok')
red_patch = mpatches.Patch(color='darkred', label='+1')
blue_patch = mpatches.Patch(color='darkblue', label='-1')
axes[0].legend(handles=[red_patch, blue_patch], loc='upper right')

# Random watermark
wm_rand_2d = random_watermark.reshape(N_BLOCKS_H, N_BLOCKS_W)
axes[1].imshow(wm_rand_2d, cmap='RdBu', vmin=-1, vmax=1, aspect='auto')
axes[1].set_title(f'Random Watermark\n(seed={SEED_RANDOM_WM})', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Kolom Blok')
axes[1].set_ylabel('Baris Blok')
axes[1].legend(handles=[red_patch, blue_patch], loc='upper right')

plt.suptitle('Visualisasi Kedua Watermark (Merah=+1, Biru=-1)', fontsize=13)
plt.tight_layout()
plt.savefig('watermarks.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# PENYISIPAN WATERMARK
# ============================================================

# Posisi mid-band dalam blok 8x8 (zigzag order, posisi 4-6)
# Koefisien ini cukup tahan kompresi tapi tidak terlalu mencolok
EMBED_POS = (2, 1)   # Posisi (baris, kolom) dalam blok 8x8 untuk penyisipan
ALPHA = 15.0          # Kekuatan watermark — makin besar makin tahan tapi makin terlihat

print(f"⚙️  Parameter Watermarking:")
print(f"   Posisi koefisien DCT : {EMBED_POS}")
print(f"   Kekuatan (alpha)     : {ALPHA}")
print(f"   Catatan: koefisien (0,0) adalah DC, makin besar indeks = frekuensi makin tinggi")


def embed_watermark(img_array, watermark, alpha=15.0, embed_pos=(2, 1)):
    """
    Sisipkan watermark ke citra melalui domain DCT.

    Langkah:
    1. Bagi citra menjadi blok 8×8
    2. Terapkan DCT ke setiap blok
    3. Tambahkan alpha * w[k] ke koefisien posisi embed_pos di blok ke-k
    4. IDCT kembali ke domain spasial

    Returns: citra ter-watermark, koefisien DCT asli (untuk ekstraksi)
    """
    h, w = img_array.shape
    pad_h = (8 - h % 8) % 8
    pad_w = (8 - w % 8) % 8
    padded = np.pad(img_array, ((0, pad_h), (0, pad_w)), mode='edge').astype(float)
    H, W = padded.shape

    result = np.zeros_like(padded)
    original_dct_coefs = []  # Simpan koefisien DCT asli untuk referensi ekstraksi

    k = 0  # Indeks watermark
    for i in range(0, H, 8):
        for j in range(0, W, 8):
            block = padded[i:i+8, j:j+8] - 128.0
            dct_block = dct2d_manual(block)

            # Simpan koefisien asli
            original_dct_coefs.append(dct_block[embed_pos])

            # Sisipkan watermark
            if k < len(watermark):
                dct_block[embed_pos] += alpha * watermark[k]
                k += 1

            result[i:i+8, j:j+8] = idct2d_manual(dct_block) + 128.0

    img_watermarked = np.clip(result[:h, :w], 0, 255).astype(np.uint8)
    return img_watermarked, np.array(original_dct_coefs)


# Sisipkan kedua watermark
img_wm_binary, orig_coefs_binary = embed_watermark(original_img, binary_watermark, ALPHA, EMBED_POS)
img_wm_random, orig_coefs_random = embed_watermark(original_img, random_watermark, ALPHA, EMBED_POS)

# Hitung PSNR (Peak Signal-to-Noise Ratio) — ukuran kualitas citra watermarked
def calc_psnr(img1, img2):
    """PSNR dalam dB. Makin tinggi = makin mirip. Di atas 40dB biasanya tidak terlihat perbedaan."""
    mse = np.mean((img1.astype(float) - img2.astype(float))**2)
    if mse == 0:
        return float('inf')
    return 20 * np.log10(255.0 / np.sqrt(mse))

psnr_bin = calc_psnr(original_img, img_wm_binary)
psnr_rand = calc_psnr(original_img, img_wm_random)

print(f"\n📊 Kualitas Citra Watermarked:")
print(f"   PSNR Binary Watermark : {psnr_bin:.2f} dB")
print(f"   PSNR Random Watermark : {psnr_rand:.2f} dB")
print(f"   (>40 dB = perbedaan tidak terlihat mata manusia)")

# Tampilkan perbandingan
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0,0].imshow(original_img, cmap='gray', vmin=0, vmax=255)
axes[0,0].set_title('Foto Asli', fontsize=12, fontweight='bold')
axes[0,0].axis('off')

axes[0,1].imshow(img_wm_binary, cmap='gray', vmin=0, vmax=255)
axes[0,1].set_title(f'+ Binary Watermark\nPSNR = {psnr_bin:.1f} dB', fontsize=12, fontweight='bold')
axes[0,1].axis('off')

axes[0,2].imshow(img_wm_random, cmap='gray', vmin=0, vmax=255)
axes[0,2].set_title(f'+ Random Watermark\nPSNR = {psnr_rand:.1f} dB', fontsize=12, fontweight='bold')
axes[0,2].axis('off')

# Perbedaan (diperbesar 5x agar terlihat)
diff_bin = np.abs(original_img.astype(int) - img_wm_binary.astype(int))
diff_rand = np.abs(original_img.astype(int) - img_wm_random.astype(int))

axes[1,0].axis('off')
axes[1,0].text(0.5, 0.5, 'Perbedaan piksel\n(diperbesar 10×\nagar terlihat)',
               ha='center', va='center', fontsize=12)

im1 = axes[1,1].imshow(np.clip(diff_bin * 10, 0, 255), cmap='hot', vmin=0, vmax=255)
axes[1,1].set_title(f'Perbedaan Binary WM × 10\nmax diff = {diff_bin.max()} piksel', fontsize=11)
axes[1,1].axis('off')
plt.colorbar(im1, ax=axes[1,1])

im2 = axes[1,2].imshow(np.clip(diff_rand * 10, 0, 255), cmap='hot', vmin=0, vmax=255)
axes[1,2].set_title(f'Perbedaan Random WM × 10\nmax diff = {diff_rand.max()} piksel', fontsize=11)
axes[1,2].axis('off')
plt.colorbar(im2, ax=axes[1,2])

plt.suptitle('Perbandingan Foto Asli vs Foto ter-Watermark', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('watermarked_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("💡 Perbedaan sangat kecil — watermark tidak terlihat secara visual!")

In [ ]:
# ============================================================
# FUNGSI EKSTRAKSI WATERMARK
# ============================================================

def extract_watermark(img_watermarked_compressed, img_original, wm_length, embed_pos=(2, 1)):
    """
    Ekstrak watermark dari citra yang sudah dikompresi.

    Metode:
    - Hitung perbedaan koefisien DCT pada posisi embed_pos
      antara citra asli dan citra ter-watermark+terkompresi
    - Sign dari perbedaan = estimasi watermark

    Returns: watermark yang diekstrak (array ±1)
    """
    h, w = img_original.shape
    pad_h = (8 - h % 8) % 8
    pad_w = (8 - w % 8) % 8

    padded_orig = np.pad(img_original, ((0, pad_h), (0, pad_w)), mode='edge').astype(float)
    padded_wm   = np.pad(img_watermarked_compressed, ((0, pad_h), (0, pad_w)), mode='edge').astype(float)
    H, W = padded_orig.shape

    extracted = []
    k = 0
    for i in range(0, H, 8):
        for j in range(0, W, 8):
            if k >= wm_length:
                break
            block_orig = padded_orig[i:i+8, j:j+8] - 128.0
            block_wm   = padded_wm[i:i+8, j:j+8] - 128.0

            dct_orig = dct2d_manual(block_orig)
            dct_wm   = dct2d_manual(block_wm)

            diff = dct_wm[embed_pos] - dct_orig[embed_pos]
            # Sign dari perbedaan = estimasi bit watermark
            extracted.append(1 if diff >= 0 else -1)
            k += 1

    return np.array(extracted)


def evaluate_watermark(original_wm, extracted_wm):
    """
    Evaluasi kualitas ekstraksi watermark.

    Returns:
    - ber  : Bit Error Rate (0 = sempurna, 1 = sama sekali salah)
    - nc   : Normalized Correlation (-1 s/d +1, mendekati 1 = bagus)
    - detected: True jika NC > 0.5 (watermark dianggap terdeteksi)
    """
    min_len = min(len(original_wm), len(extracted_wm))
    orig = original_wm[:min_len]
    extr = extracted_wm[:min_len]

    # BER: proporsi bit yang berbeda
    ber = np.sum(orig != extr) / min_len

    # NC: korelasi ternormalisasi
    nc = np.dot(orig.astype(float), extr.astype(float)) / (np.linalg.norm(orig) * np.linalg.norm(extr) + 1e-10)

    detected = nc > 0.5
    return ber, nc, detected


# Test ekstraksi tanpa kompresi (sebagai baseline)
ext_bin_nocomp = extract_watermark(img_wm_binary, original_img, WM_SIZE, EMBED_POS)
ext_rand_nocomp = extract_watermark(img_wm_random, original_img, WM_SIZE, EMBED_POS)

ber_b, nc_b, det_b = evaluate_watermark(binary_watermark, ext_bin_nocomp)
ber_r, nc_r, det_r = evaluate_watermark(random_watermark, ext_rand_nocomp)

print("🧪 Baseline: Ekstraksi TANPA kompresi")
print(f"   Binary WM  — BER: {ber_b:.4f}, NC: {nc_b:.4f}, Terdeteksi: {'✅ Ya' if det_b else '❌ Tidak'}")
print(f"   Random WM  — BER: {ber_r:.4f}, NC: {nc_r:.4f}, Terdeteksi: {'✅ Ya' if det_r else '❌ Tidak'}")
print()
print("💡 Idealnya BER=0 dan NC=1 jika tidak ada gangguan sama sekali.")
print("   Nilai BER/NC sedikit berbeda dari ideal karena IDCT→round→DCT menambah noise kecil.")

In [ ]:
# ============================================================
# FUNGSI EKSTRAKSI WATERMARK
# ============================================================

def extract_watermark(img_watermarked_compressed, img_original, wm_length, embed_pos=(2, 1)):
    """
    Ekstrak watermark dari citra yang sudah dikompresi.

    Metode:
    - Hitung perbedaan koefisien DCT pada posisi embed_pos
      antara citra asli dan citra ter-watermark+terkompresi
    - Sign dari perbedaan = estimasi watermark

    Returns: watermark yang diekstrak (array ±1)
    """
    h, w = img_original.shape
    pad_h = (8 - h % 8) % 8
    pad_w = (8 - w % 8) % 8

    padded_orig = np.pad(img_original, ((0, pad_h), (0, pad_w)), mode='edge').astype(float)
    padded_wm   = np.pad(img_watermarked_compressed, ((0, pad_h), (0, pad_w)), mode='edge').astype(float)
    H, W = padded_orig.shape

    extracted = []
    k = 0
    for i in range(0, H, 8):
        for j in range(0, W, 8):
            if k >= wm_length:
                break
            block_orig = padded_orig[i:i+8, j:j+8] - 128.0
            block_wm   = padded_wm[i:i+8, j:j+8] - 128.0

            dct_orig = dct2d_manual(block_orig)
            dct_wm   = dct2d_manual(block_wm)

            diff = dct_wm[embed_pos] - dct_orig[embed_pos]
            # Sign dari perbedaan = estimasi bit watermark
            extracted.append(1 if diff >= 0 else -1)
            k += 1

    return np.array(extracted)


def evaluate_watermark(original_wm, extracted_wm):
    """
    Evaluasi kualitas ekstraksi watermark.

    Returns:
    - ber  : Bit Error Rate (0 = sempurna, 1 = sama sekali salah)
    - nc   : Normalized Correlation (-1 s/d +1, mendekati 1 = bagus)
    - detected: True jika NC > 0.5 (watermark dianggap terdeteksi)
    """
    min_len = min(len(original_wm), len(extracted_wm))
    orig = original_wm[:min_len]
    extr = extracted_wm[:min_len]

    # BER: proporsi bit yang berbeda
    ber = np.sum(orig != extr) / min_len

    # NC: korelasi ternormalisasi
    nc = np.dot(orig.astype(float), extr.astype(float)) / (np.linalg.norm(orig) * np.linalg.norm(extr) + 1e-10)

    detected = nc > 0.5
    return ber, nc, detected


# Test ekstraksi tanpa kompresi (sebagai baseline)
ext_bin_nocomp = extract_watermark(img_wm_binary, original_img, WM_SIZE, EMBED_POS)
ext_rand_nocomp = extract_watermark(img_wm_random, original_img, WM_SIZE, EMBED_POS)

ber_b, nc_b, det_b = evaluate_watermark(binary_watermark, ext_bin_nocomp)
ber_r, nc_r, det_r = evaluate_watermark(random_watermark, ext_rand_nocomp)

print("🧪 Baseline: Ekstraksi TANPA kompresi")
print(f"   Binary WM  — BER: {ber_b:.4f}, NC: {nc_b:.4f}, Terdeteksi: {'✅ Ya' if det_b else '❌ Tidak'}")
print(f"   Random WM  — BER: {ber_r:.4f}, NC: {nc_r:.4f}, Terdeteksi: {'✅ Ya' if det_r else '❌ Tidak'}")
print()
print("💡 Idealnya BER=0 dan NC=1 jika tidak ada gangguan sama sekali.")
print("   Nilai BER/NC sedikit berbeda dari ideal karena IDCT→round→DCT menambah noise kecil.")

In [ ]:
# ============================================================
# EVALUASI DENGAN BERBAGAI QUALITY FACTOR
# ============================================================

# Rentang QF yang akan diuji
QUALITY_FACTORS = [95, 90, 80, 70, 60, 50, 40, 30, 20, 10, 5]

results_binary = {'qf': [], 'ber': [], 'nc': [], 'detected': [], 'psnr_img': []}
results_random = {'qf': [], 'ber': [], 'nc': [], 'detected': [], 'psnr_img': []}

print("🔄 Memulai evaluasi... (ini mungkin memakan waktu beberapa menit)")
print("-" * 75)
print(f"{'QF':>4} | {'PSNR Img':>9} | {'Binary BER':>10} | {'Binary NC':>9} | {'Rand BER':>8} | {'Rand NC':>8}")
print("-" * 75)

for qf in QUALITY_FACTORS:
    # Kompres citra binary-watermarked
    compressed_bin = jpeg_compress_decompress(img_wm_binary, qf)
    # Kompres citra random-watermarked
    compressed_rand = jpeg_compress_decompress(img_wm_random, qf)

    # Hitung PSNR citra (kualitas citra setelah kompresi)
    psnr_img = calc_psnr(img_wm_binary, compressed_bin)

    # Ekstrak watermark dari citra terkompresi
    ext_bin  = extract_watermark(compressed_bin,  original_img, WM_SIZE, EMBED_POS)
    ext_rand = extract_watermark(compressed_rand, original_img, WM_SIZE, EMBED_POS)

    # Evaluasi
    ber_b, nc_b, det_b = evaluate_watermark(binary_watermark, ext_bin)
    ber_r, nc_r, det_r = evaluate_watermark(random_watermark, ext_rand)

    # Simpan hasil
    for res, ber, nc, det in [(results_binary, ber_b, nc_b, det_b),
                               (results_random, ber_r, nc_r, det_r)]:
        res['qf'].append(qf)
        res['ber'].append(ber)
        res['nc'].append(nc)
        res['detected'].append(det)
        res['psnr_img'].append(psnr_img)

    status_b = '✅' if det_b else '❌'
    status_r = '✅' if det_r else '❌'
    print(f"{qf:>4} | {psnr_img:>8.1f}dB | {ber_b:>9.4f}{status_b} | {nc_b:>9.4f} | {ber_r:>7.4f}{status_r} | {nc_r:>8.4f}")

print("-" * 75)
print("Keterangan: ✅ = watermark TERDETEKSI (NC > 0.5), ❌ = watermark GAGAL DIEKSTRAK")

In [ ]:
# ============================================================
# VISUALISASI CITRA TERKOMPRESI PADA BEBERAPA QF
# ============================================================

showcase_qfs = [95, 50, 20, 5]
fig, axes = plt.subplots(2, len(showcase_qfs) + 1, figsize=(18, 8))

# Kolom pertama: foto ter-watermark
axes[0, 0].imshow(img_wm_binary, cmap='gray', vmin=0, vmax=255)
axes[0, 0].set_title('Foto + Binary WM\n(sebelum kompresi)', fontsize=10, fontweight='bold')
axes[0, 0].axis('off')

axes[1, 0].imshow(img_wm_random, cmap='gray', vmin=0, vmax=255)
axes[1, 0].set_title('Foto + Random WM\n(sebelum kompresi)', fontsize=10, fontweight='bold')
axes[1, 0].axis('off')

for col, qf in enumerate(showcase_qfs, start=1):
    # Kompres
    comp_b = jpeg_compress_decompress(img_wm_binary, qf)
    comp_r = jpeg_compress_decompress(img_wm_random, qf)

    # Ekstrak
    ext_b = extract_watermark(comp_b, original_img, WM_SIZE, EMBED_POS)
    ext_r = extract_watermark(comp_r, original_img, WM_SIZE, EMBED_POS)
    _, nc_b, det_b = evaluate_watermark(binary_watermark, ext_b)
    _, nc_r, det_r = evaluate_watermark(random_watermark, ext_r)

    psnr_b = calc_psnr(img_wm_binary, comp_b)
    psnr_r = calc_psnr(img_wm_random, comp_r)

    status_b = '✅ Terdeteksi' if det_b else '❌ Gagal'
    status_r = '✅ Terdeteksi' if det_r else '❌ Gagal'

    axes[0, col].imshow(comp_b, cmap='gray', vmin=0, vmax=255)
    axes[0, col].set_title(f'QF={qf}\nPSNR={psnr_b:.1f}dB\nNC={nc_b:.3f} {status_b}',
                            fontsize=9, fontweight='bold',
                            color='darkgreen' if det_b else 'darkred')
    axes[0, col].axis('off')

    axes[1, col].imshow(comp_r, cmap='gray', vmin=0, vmax=255)
    axes[1, col].set_title(f'QF={qf}\nPSNR={psnr_r:.1f}dB\nNC={nc_r:.3f} {status_r}',
                            fontsize=9, fontweight='bold',
                            color='darkgreen' if det_r else 'darkred')
    axes[1, col].axis('off')

plt.suptitle('Citra Terkompresi pada Berbagai QF\nBaris atas: Binary WM | Baris bawah: Random WM',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('compressed_images.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# ANALISIS QF KRITIS
# ============================================================

print("=" * 60)
print("📋 RINGKASAN HASIL EVALUASI WATERMARKING")
print("=" * 60)
print()

# Binary watermark
print("🔵 BINARY WATERMARK:")
crit_qf_bin = None
for qf, det, ber, nc in zip(results_binary['qf'], results_binary['detected'],
                              results_binary['ber'], results_binary['nc']):
    status = "TERDETEKSI ✅" if det else "GAGAL ❌"
    print(f"   QF={qf:3d}: BER={ber:.4f}, NC={nc:.4f} → {status}")
    if not det and crit_qf_bin is None:
        crit_qf_bin = qf

print()
if crit_qf_bin:
    print(f"   ⚠️  Binary WM GAGAL diekstrak mulai QF = {crit_qf_bin}")
else:
    print(f"   ✅ Binary WM berhasil diekstrak di semua QF yang diuji!")

print()
print("🔴 RANDOM WATERMARK:")
crit_qf_rand = None
for qf, det, ber, nc in zip(results_random['qf'], results_random['detected'],
                              results_random['ber'], results_random['nc']):
    status = "TERDETEKSI ✅" if det else "GAGAL ❌"
    print(f"   QF={qf:3d}: BER={ber:.4f}, NC={nc:.4f} → {status}")
    if not det and crit_qf_rand is None:
        crit_qf_rand = qf

print()
if crit_qf_rand:
    print(f"   ⚠️  Random WM GAGAL diekstrak mulai QF = {crit_qf_rand}")
else:
    print(f"   ✅ Random WM berhasil diekstrak di semua QF yang diuji!")

print()
print("=" * 60)
print("📌 KESIMPULAN:")
print("=" * 60)
print()
print("Watermark disisipkan di koefisien DCT mid-band (posisi 2,1)")
print(f"dengan kekuatan alpha = {ALPHA}.")
print()
if crit_qf_bin:
    print(f"• Binary Watermark: tidak dapat diekstrak saat QF ≤ {crit_qf_bin}")
    print(f"  → Kompresi JPEG dengan QF ≤ {crit_qf_bin} menghancurkan pola biner yang tersimpan.")
if crit_qf_rand:
    print(f"• Random Watermark: tidak dapat diekstrak saat QF ≤ {crit_qf_rand}")
    print(f"  → Meski acak, koefisien DCT juga dihapus oleh kuantisasi agresif.")
print()
print("Mengapa ini terjadi?")
print("Pada QF rendah, nilai tabel kuantisasi menjadi sangat besar.")
print("Koefisien DCT yang kecil (termasuk yang mengandung watermark)")
print("dibulatkan ke 0 saat kuantisasi → informasi watermark hilang.")

In [ ]:
# ============================================================
# ANALISIS KOEFISIEN DCT PADA BERBAGAI QF
# ============================================================

# Ambil satu blok representatif dari tengah citra
block_row, block_col = N_BLOCKS_H // 2, N_BLOCKS_W // 2
br, bc = block_row * 8, block_col * 8

block_orig = original_img[br:br+8, bc:bc+8].astype(float) - 128
block_wm_b = img_wm_binary[br:br+8, bc:bc+8].astype(float) - 128

dct_orig_block = dct2d_manual(block_orig)
dct_wm_block   = dct2d_manual(block_wm_b)

fig, axes = plt.subplots(2, 4, figsize=(18, 8))

# QF untuk analisis blok
analysis_qfs = [95, 50, 20, 5]

# Baris 1: koefisien DCT setelah kuantisasi
for col, qf in enumerate(analysis_qfs):
    qt = get_quant_table(qf)

    # Kuantisasi blok watermarked
    q_block = quantize_block(dct_wm_block, qt)

    im = axes[0, col].imshow(np.abs(q_block), cmap='YlOrRd', aspect='auto')
    axes[0, col].set_title(f'Koefisien DCT (|nilai|)\nsetelah kuantisasi QF={qf}', fontsize=10)
    plt.colorbar(im, ax=axes[0, col])

    # Tandai posisi watermark
    r, c = EMBED_POS
    axes[0, col].add_patch(plt.Rectangle((c-0.5, r-0.5), 1, 1, fill=False,
                                          edgecolor='blue', linewidth=3))
    val = q_block[r, c]
    axes[0, col].text(c, r, f'{int(val)}', ha='center', va='center',
                      fontsize=11, fontweight='bold', color='blue')

    # Hitung berapa % koefisien yang menjadi 0
    zero_pct = 100 * np.sum(q_block == 0) / 64
    axes[0, col].set_xlabel(f'{zero_pct:.1f}% koefisien = 0', fontsize=9, color='red')
    axes[0, col].set_xticks([])
    axes[0, col].set_yticks([])

# Baris 2: perbandingan koefisien asli vs after JPEG cycle di posisi watermark
embed_coefs_after_jpeg = []
all_qfs_dense = list(range(5, 100, 2))

for qf in all_qfs_dense:
    qt = get_quant_table(qf)
    q = quantize_block(dct_wm_block, qt)[EMBED_POS]
    dq = dequantize_block(q, qt[EMBED_POS[0], EMBED_POS[1]])
    embed_coefs_after_jpeg.append(dq)

ax2 = axes[1, 0]
ax2.plot(all_qfs_dense, embed_coefs_after_jpeg, 'b-', linewidth=2, label='Koefisien setelah JPEG cycle')
ax2.axhline(y=dct_wm_block[EMBED_POS], color='green', linestyle='--', linewidth=2, label=f'Koefisien asli={dct_wm_block[EMBED_POS]:.1f}')
ax2.axhline(y=dct_orig_block[EMBED_POS], color='orange', linestyle='--', linewidth=2, label=f'Tanpa WM={dct_orig_block[EMBED_POS]:.1f}')
ax2.axhline(y=0, color='red', linestyle=':', linewidth=1.5, label='Nol (watermark hilang)')
ax2.set_xlabel('Quality Factor', fontsize=11)
ax2.set_ylabel(f'Koefisien DCT di posisi {EMBED_POS}', fontsize=10)
ax2.set_title('Nilai Koefisien WM vs QF\n(blok tengah)', fontsize=11, fontweight='bold')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

# Proporsi koefisien = 0 vs QF
ax3 = axes[1, 1]
zero_proportions = []
for qf in all_qfs_dense:
    qt = get_quant_table(qf)
    q = quantize_block(dct_wm_block, qt)
    zero_proportions.append(100 * np.sum(q == 0) / 64)

ax3.plot(all_qfs_dense, zero_proportions, 'r-', linewidth=2)
ax3.fill_between(all_qfs_dense, zero_proportions, alpha=0.3, color='red')
ax3.set_xlabel('Quality Factor', fontsize=11)
ax3.set_ylabel('% Koefisien DCT = 0', fontsize=11)
ax3.set_title('Proporsi Koefisien Hilang\nvs Quality Factor', fontsize=11, fontweight='bold')
ax3.grid(True, alpha=0.3)

# Penjelasan tabel kuantisasi
ax4 = axes[1, 2]
qf_vals = [5, 10, 20, 30, 50, 70, 90, 95]
qt_at_embed = [get_quant_table(q)[EMBED_POS] for q in qf_vals]
bars = ax4.bar([str(q) for q in qf_vals], qt_at_embed, color=['#d32f2f','#e64a19','#f57c00','#ffa000','#388e3c','#1976d2','#7b1fa2','#4a148c'])
ax4.set_xlabel('Quality Factor', fontsize=11)
ax4.set_ylabel(f'Nilai Kuantisasi di posisi {EMBED_POS}', fontsize=10)
ax4.set_title(f'Nilai Tabel Kuantisasi di Posisi {EMBED_POS}\nvs QF', fontsize=11, fontweight='bold')
for bar, val in zip(bars, qt_at_embed):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{int(val)}',
             ha='center', fontsize=10, fontweight='bold')
ax4.grid(True, alpha=0.3, axis='y')

axes[1, 3].axis('off')
axes[1, 3].text(0.5, 0.5,
    f'KESIMPULAN:\n\n'
    f'Nilai alpha (kekuatan WM):\n{ALPHA}\n\n'
    f'Posisi koefisien: {EMBED_POS}\n\n'
    f'Watermark hilang jika:\n'
    f'nilai kuantisasi > alpha\n\n'
    f'Karena:\nround(coef / qt) = 0\n→ informasi WM hilang!',
    ha='center', va='center', fontsize=11,
    bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.suptitle('Analisis Detail: Mengapa Watermark Bisa Hilang?', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('dct_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print("💡 Kunci: watermark hilang ketika nilai kuantisasi > alpha (kekuatan watermark).")
print(f"   Di posisi {EMBED_POS}: nilai kuantisasi saat QF=50 adalah {int(get_quant_table(50)[EMBED_POS])}")
print(f"   Alpha kita = {ALPHA}. Jika kuantisasi > {ALPHA}, koefisien dibulatkan ke 0!")

In [ ]:
# ============================================================
# TABEL RINGKASAN AKHIR
# ============================================================

print("╔" + "═"*62 + "╗")
print("║" + " RINGKASAN EKSPERIMEN WATERMARKING ".center(62) + "║")
print("╠" + "═"*62 + "╣")
print(f"║  Ukuran citra        : {IMG_SIZE}×{IMG_SIZE} piksel".ljust(63) + "║")
print(f"║  Jumlah blok 8×8     : {WM_SIZE} blok".ljust(63) + "║")
print(f"║  Posisi embedding    : DCT koefisien {EMBED_POS}".ljust(63) + "║")
print(f"║  Kekuatan watermark  : alpha = {ALPHA}".ljust(63) + "║")
print(f"║  PSNR Binary WM      : {psnr_bin:.2f} dB".ljust(63) + "║")
print(f"║  PSNR Random WM      : {psnr_rand:.2f} dB".ljust(63) + "║")
print("╠" + "═"*62 + "╣")
print("║" + " HASIL EVALUASI ".center(62) + "║")
print("╠" + "═"*20 + "╦" + "═"*20 + "╦" + "═"*20 + "╣")
print("║" + " QF ".center(20) + "║" + " Binary WM ".center(20) + "║" + " Random WM ".center(20) + "║")
print("╠" + "═"*20 + "╬" + "═"*20 + "╬" + "═"*20 + "╣")

for qf, det_b, nc_b, det_r, nc_r in zip(
        results_binary['qf'],
        results_binary['detected'], results_binary['nc'],
        results_random['detected'], results_random['nc']):
    sb = f"✅ NC={nc_b:.3f}" if det_b else f"❌ NC={nc_b:.3f}"
    sr = f"✅ NC={nc_r:.3f}" if det_r else f"❌ NC={nc_r:.3f}"
    print(f"║{str(qf).center(20)}║{sb.center(20)}║{sr.center(20)}║")

print("╚" + "═"*20 + "╩" + "═"*20 + "╩" + "═"*20 + "╝")
print()
print("Keterangan: ✅ = Watermark TERDETEKSI (NC > 0.5)")
print("            ❌ = Watermark GAGAL diekstrak (NC ≤ 0.5)")
print()
print("✅ Eksperimen selesai! Semua output disimpan sebagai file PNG.")